**IMPORTING OF LIBRARIES**

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import cv2
from matplotlib import gridspec
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory, image
from tensorflow.keras.applications import ResNet50, MobileNetV2, VGG16
from tensorflow.keras.models import Sequential, Model, load_model, clone_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Concatenate, Conv2D, MaxPooling2D
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomContrast
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, f1_score, ConfusionMatrixDisplay, roc_curve, auc


**DATA PREPROCESSING**

In [ ]:
dataset_path = "/kaggle/input/chest-x-rays/Dataset of Tuberculosis Chest X-rays Images"

dataset = image_dataset_from_directory(
    dataset_path,
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'  
)


In [ ]:
TB_count = len(os.listdir(os.path.join(dataset_path, "TB Chest X-rays")))
Normal_count = len(os.listdir(os.path.join(dataset_path, "Normal Chest X-rays")))

print(f"TB images: {TB_count}")
print(f"Normal images: {Normal_count}")

In [ ]:
dataset = image_dataset_from_directory(
    dataset_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True,
    seed=42,
    label_mode='binary'
)

In [ ]:
dataset_size = 0
for _ in dataset:
    dataset_size += 1

total_samples = dataset.cardinality().numpy() * 32  


In [ ]:
train_size = int(0.7 * dataset_size)
val_size = int(0.2 * dataset_size)
test_size = dataset_size - train_size - val_size

train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size).take(val_size)
test_ds = dataset.skip(train_size + val_size)


In [ ]:
num_train = len(list(train_ds.unbatch()))
num_val = len(list(val_ds.unbatch()))
num_test = len(list(test_ds.unbatch()))

print(f"Training images: {num_train}")
print(f"Validation images: {num_val}")
print(f"Test images: {num_test}")


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

In [ ]:
num_train = len(list(train_ds.unbatch()))
num_val = len(list(val_ds.unbatch()))
num_test = len(list(test_ds.unbatch()))

print(f"Training images: {num_train}")
print(f"Validation images: {num_val}")
print(f"Test images: {num_test}")


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
for images, labels in train_ds.take(1):
    plt.figure(figsize=(9, 9))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8).numpy())  # If needed
        plt.title(f"Label: {int(labels[i])}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

**ENSEMBLE MODEL BUILD**

In [ ]:
input_tensor = Input(shape=(224, 224, 3))

resnet = ResNet50(weights='imagenet', include_top=False, input_tensor=input_tensor)
vgg = VGG16(weights='imagenet', include_top=False, input_tensor=input_tensor)
mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_tensor=input_tensor)

for layer in resnet.layers[-20:]:
    layer.trainable = True
for layer in vgg.layers[-20:]:
    layer.trainable = True
for layer in mobilenet.layers[-20:]:
    layer.trainable = True

resnet_out = GlobalAveragePooling2D()(resnet.output)
vgg_out = GlobalAveragePooling2D()(vgg.output)
mobilenet_out = GlobalAveragePooling2D()(mobilenet.output)

merged = Concatenate()([resnet_out, vgg_out, mobilenet_out])
merged = Dense(256, activation='relu')(merged)
merged = Dropout(0.5)(merged)
output = Dense(1, activation='sigmoid')(merged)

fine_tuned_model = Model(inputs=input_tensor, outputs=output)

fine_tuned_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

fine_tune_history = fine_tuned_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)

In [ ]:
fine_tuned_model.save("/kaggle/working/fine_tuned_ensemble.keras")
resnet.save("/kaggle/working/resnet.keras")
vgg.save("/kaggle/working/vgg.keras")
mobilenet.save("/kaggle/working/mobile.keras")
with open("/kaggle/working/fine_tune_history.pkl", "wb") as f:
    pickle.dump(fine_tune_history.history, f)

In [ ]:
fine_tuned_model = load_model("/kaggle/working/fine_tuned_ensemble.keras")
resnet = load_model("/kaggle/working/resnet.keras")
vgg = load_model("/kaggle/working/vgg.keras")
mobilenet = load_model("/kaggle/working/mobile.keras")
with open("/kaggle/working/fine_tune_history.pkl", "rb") as f:
    history = pickle.load(f)

**ENSEMBLE MODEL EVALUATION**

In [ ]:
fine_tuned_model.evaluate(test_ds)

In [ ]:
plt.plot(fine_tune_history.history['accuracy'], label='Train Accuracy')
plt.plot(fine_tune_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.plot(fine_tune_history.history['loss'], label='Train Loss')
plt.plot(fine_tune_history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = fine_tuned_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

class_names = ['Normal', 'Tuberculosis']
class_result = classification_report(y_true, y_pred, target_names=class_names)
print(class_result)

In [ ]:
class_names = ['Normal', 'Tuberculosis']
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = fine_tuned_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

**ENSEMBLE MODEL TESTCASE**

In [ ]:
img_path = "/kaggle/input/tryinger/let's try/Tuberculosis-654.png"

img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

prediction = fine_tuned_model.predict(img_array)

predicted_class = 'Tuberculosis' if prediction[0][0] > 0.5 else 'Normal'
confidence = prediction[0][0] if predicted_class == 'Tuberculosis' else 1 - prediction[0][0]

print(f"Predicted Class: {predicted_class}")
print(f"Confidence: {confidence:.2%}")

**ENSEMBLE MODEL GRADCAM IMPLEMENTATION**

In [ ]:
def gradcam_for_model(img_array, model, layer_name, target_size=(224, 224)):
    grad_model = tf.keras.models.Model([model.input], [model.get_layer(layer_name).output, model.output])

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)[0]
    conv_outputs = conv_outputs[0]

    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.reduce_sum(tf.multiply(weights, conv_outputs), axis=-1).numpy()
    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)
    cam = cv2.resize(cam, target_size)
    return cam


def generate_dynamic_gradcam(img_path, resnet, vgg, mobilenet, ensemble_model):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    resnet_head = tf.keras.Sequential([
        resnet,
        GlobalAveragePooling2D(),
        Dense(1, activation='sigmoid')
    ])

    vgg_head = tf.keras.Sequential([
        vgg,
        GlobalAveragePooling2D(),
        Dense(1, activation='sigmoid')
    ])

    mobilenet_head = tf.keras.Sequential([
        mobilenet,
        GlobalAveragePooling2D(),
        Dense(1, activation='sigmoid')
    ])

    resnet_pred = resnet_head.predict(img_array)
    vgg_pred = vgg_head.predict(img_array)
    mobilenet_pred = mobilenet_head.predict(img_array)

    resnet_conf = float(resnet_pred[0][0])
    vgg_conf = float(vgg_pred[0][0])
    mobilenet_conf = float(mobilenet_pred[0][0])

    confidences = [
        resnet_conf if resnet_conf > 0.5 else 1 - resnet_conf,
        vgg_conf if vgg_conf > 0.5 else 1 - vgg_conf,
        mobilenet_conf if mobilenet_conf > 0.5 else 1 - mobilenet_conf
    ]

    models = [resnet, vgg, mobilenet]
    layer_names = ['conv5_block3_out', 'block5_conv3', 'Conv_1']  

    best_idx = np.argmax(confidences)
    selected_model = models[best_idx]
    selected_layer = layer_names[best_idx]

    cam = gradcam_for_model(img_array, selected_model, selected_layer)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)

    img_display = np.uint8(img_array[0])
    overlay = cv2.addWeighted(img_display, 0.6, heatmap, 0.4, 0)

    pred = ensemble_model.predict(img_array)
    predicted_class = 'Tuberculosis' if pred[0][0] > 0.5 else 'Normal'
    confidence = pred[0][0] if predicted_class == 'Tuberculosis' else 1 - pred[0][0]

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(np.array(img).astype('uint8'))
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    model_name = ['ResNet50', 'VGG16', 'MobileNetV2'][best_idx]
    plt.title(f"{model_name} Grad-CAM\n{predicted_class} ({confidence:.2%})")
    plt.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
img_path = "/kaggle/input/testinger/TB.1.jpg"

generate_dynamic_gradcam(
    img_path=img_path,
    resnet=resnet,
    vgg=vgg,
    mobilenet=mobilenet,
    ensemble_model=fine_tuned_model  
)
